In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# SE Block
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_channels, in_channels // reduction, bias=False)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(in_channels // reduction, in_channels, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.shape
        squeeze = self.global_avg_pool(x).view(b, c)
        excitation = self.fc1(squeeze)
        excitation = self.relu(excitation)
        excitation = self.fc2(excitation)
        excitation = self.sigmoid(excitation).view(b, c, 1, 1)
        return x * excitation  

# RCH Block with SE Block
class RCHBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size1=15, kernel_size2=7, final_layer=False):
        super().__init__()
        padding1 = (kernel_size1 - 1) // 2  
        padding2 = (kernel_size2 - 1) // 2  
        
        # Parallel Convolution Blocks
        self.block1 = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size1, stride=1, padding=padding1)
        self.block2 = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size1, stride=1, padding=padding1)
        
        # Additional 1x1 Convolution
        self.block3 = nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0)
        
        # Batch Normalization
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # SE Attention Block
        self.se = SEBlock(out_channels)

        # Sigmoid Activation
        self.final_layer = final_layer

    def forward(self, x):
        x1 = self.block1(x)
        x2 = self.block2(x)
        
        # Ensure x1 and x2 are the same shape before addition
        if x1.shape != x2.shape:
            x2 = F.interpolate(x2, size=x1.shape[2:], mode='bilinear', align_corners=False)

        # Pass x1 through additional conv block
        x3 = self.block3(x1)

        # Normalize
        x1, x3 = self.bn1(x1), self.bn2(x3)

        # Add both paths
        out = x1 + x3

        # Apply SE Attention
        out = self.se(out)

        # Sigmoid Activation
        out = torch.sigmoid(out)

        return out

# RCH Classifier
class RCHClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        
        self.rch1 = RCHBlock(in_channels=3, out_channels=64, kernel_size1=5)
        self.pool1 = nn.AvgPool2d(2)  
        
        self.rch2 = RCHBlock(in_channels=64, out_channels=128, kernel_size1=5)
        self.pool2 = nn.AvgPool2d(2)  

        self.rch3 = RCHBlock(in_channels=128, out_channels=256, kernel_size1=5)
        self.pool3 = nn.AdaptiveAvgPool2d(1)  

        self.fc = nn.Linear(256, num_classes) 

    def forward(self, x):
        x = self.pool1(self.rch1(x))  
        x = self.pool2(self.rch2(x))  
        x = self.pool3(self.rch3(x))  

        x = torch.flatten(x, 1)  
        x = self.fc(x)  
        
        return x

# Model Summary
from torchsummary import summary

device = "cuda" if torch.cuda.is_available() else "cpu"
model = RCHClassifier(num_classes=2).to(device)

summary(model, (3, 256, 256))  


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 256, 256]           4,864
            Conv2d-2         [-1, 64, 256, 256]           4,864
            Conv2d-3         [-1, 64, 256, 256]           4,160
       BatchNorm2d-4         [-1, 64, 256, 256]             128
       BatchNorm2d-5         [-1, 64, 256, 256]             128
 AdaptiveAvgPool2d-6             [-1, 64, 1, 1]               0
            Linear-7                    [-1, 4]             256
              ReLU-8                    [-1, 4]               0
            Linear-9                   [-1, 64]             256
          Sigmoid-10                   [-1, 64]               0
          SEBlock-11         [-1, 64, 256, 256]               0
         RCHBlock-12         [-1, 64, 256, 256]               0
        AvgPool2d-13         [-1, 64, 128, 128]               0
           Conv2d-14        [-1, 128, 1

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import make_grid

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((64, 64)),
    transforms.Normalize((0.5,), (0.5,))
])

batch_size = 10

train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)

val_data = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=batch_size, shuffle=False)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 
           'dog', 'frog', 'horse', 'ship', 'truck')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RCHClassifier(num_classes=10)

criterion = nn.CrossEntropyLoss()  # Suitable for classification
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

num_epochs = 200
train_losses, val_losses, val_accuracies = [], [], []
best_val_loss = float("inf")

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images, labels

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_accuracy = correct / total * 100
    train_losses.append(train_loss)

       model.eval()
    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images, labels
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_accuracy = correct / total * 100
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_rch_model.pth")

    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")


plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(num_epochs), train_losses, label="Train Loss")
plt.plot(range(num_epochs), val_losses, label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(num_epochs), val_accuracies, label="Val Accuracy", color="red")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy")
plt.legend()
plt.show()


Files already downloaded and verified
Files already downloaded and verified
Epoch [1/200], Train Loss: 1.9010, Val Loss: 2.4666, Val Acc: 21.49%
Epoch [2/200], Train Loss: 1.5800, Val Loss: 1.5702, Val Acc: 41.98%
Epoch [3/200], Train Loss: 1.3786, Val Loss: 1.5467, Val Acc: 44.10%
Epoch [4/200], Train Loss: 1.2536, Val Loss: 1.4633, Val Acc: 46.89%
Epoch [5/200], Train Loss: 1.1687, Val Loss: 1.1557, Val Acc: 59.14%
Epoch [6/200], Train Loss: 1.0941, Val Loss: 1.3097, Val Acc: 54.35%
Epoch [7/200], Train Loss: 1.0452, Val Loss: 1.1007, Val Acc: 60.63%
Epoch [8/200], Train Loss: 0.9971, Val Loss: 1.0397, Val Acc: 63.36%
Epoch [9/200], Train Loss: 0.9535, Val Loss: 0.9905, Val Acc: 63.96%
Epoch [10/200], Train Loss: 0.9208, Val Loss: 1.2096, Val Acc: 58.03%
Epoch [11/200], Train Loss: 0.8871, Val Loss: 1.0518, Val Acc: 62.68%
Epoch [12/200], Train Loss: 0.8668, Val Loss: 1.1381, Val Acc: 61.83%
Epoch [13/200], Train Loss: 0.8310, Val Loss: 0.9318, Val Acc: 67.42%
Epoch [14/200], Train L

KeyboardInterrupt: 